In [1]:
import xarray as xr
import pandas as pd
import os

# --- Load dataset ---
path = "/storage/ice-shared/bmed6780/climate/na_cordex_monthly_2000_2020.nc"
ds = xr.open_dataset(path)

# --- Define cities with coordinates ---
cities = {
    "Atlanta": (33.75, -84.39),
    "Los_Angeles": (34.05, -118.25),
    "Chicago": (41.88, -87.63),
    "San_Francisco": (37.77, -122.42),
    "New_York": (40.71, -74.01),
    "St_Louis": (38.63, -90.20)
}

# --- Output directory (change if needed) ---
output_dir = "/home/hice1/jblack74/scratch/BHI/CORDEX_Dataset"
os.makedirs(output_dir, exist_ok=True)

# --- Loop through cities ---
for city, (lat_city, lon_city) in cities.items():
    
    # Extract nearest grid point
    tas = ds['tas'].sel(lat=lat_city, lon=lon_city, method='nearest')
    hurs = ds['hurs'].sel(lat=lat_city, lon=lon_city, method='nearest')
    ps = ds['ps'].sel(lat=lat_city, lon=lon_city, method='nearest')
    
    # Align time dimension
    tas, hurs, ps = xr.align(tas, hurs, ps)
    
    # Convert to 1D arrays
    tas_vals = tas.squeeze().values.flatten()
    hurs_vals = hurs.squeeze().values.flatten()
    ps_vals = ps.squeeze().values.flatten()
    times = pd.to_datetime(tas['time'].values)
    
    # Ensure equal lengths
    min_len = min(len(times), len(tas_vals), len(hurs_vals), len(ps_vals))
    
    df = pd.DataFrame({
        'time': times[:min_len],
        'tas': tas_vals[:min_len],
        'hurs': hurs_vals[:min_len],
        'ps': ps_vals[:min_len]
    })
    
    # Save CSV
    filename = os.path.join(output_dir, f"{city}_climate.csv")
    df.to_csv(filename, index=False)
    
    print(f"Saved {city} data to {filename}")

Saved Atlanta data to /home/hice1/jblack74/scratch/BHI/CORDEX_Dataset/Atlanta_climate.csv
Saved Los_Angeles data to /home/hice1/jblack74/scratch/BHI/CORDEX_Dataset/Los_Angeles_climate.csv
Saved Chicago data to /home/hice1/jblack74/scratch/BHI/CORDEX_Dataset/Chicago_climate.csv
Saved San_Francisco data to /home/hice1/jblack74/scratch/BHI/CORDEX_Dataset/San_Francisco_climate.csv
Saved New_York data to /home/hice1/jblack74/scratch/BHI/CORDEX_Dataset/New_York_climate.csv
Saved St_Louis data to /home/hice1/jblack74/scratch/BHI/CORDEX_Dataset/St_Louis_climate.csv
